# 📄 Topic 2: NLP Preprocessing for RAG

## What I'm going to show you

Raw PDF text is messy. Headers bleed into body text. Page numbers appear mid-sentence.
Whitespace is inconsistent. If you feed that raw noise directly into an LLM, it struggles
to find the signal. I've seen assistants confidently answer from a page header instead of
the actual product terms.

In this notebook I'm going to show you how to:

- Load Barclays product documents from S3 using boto3 and PyMuPDF
- Clean the extracted text so it is consistent and noise-free
- Chunk the clean text into pieces the right size for a retrieval pipeline

By the end, you will have a `chunks` list of clean document fragments ready to embed
into a vector store in Topic 6.

## Learning Objectives

By the end of this topic you will be able to:

1. Extract text from a PDF using PyMuPDF (`import pymupdf`)
2. Clean raw extracted text with regex and normalization
3. Implement fixed-size chunking with overlap
4. Explain the tradeoff between chunk size and retrieval quality

## What component of the assistant am I building today?

The **Document Ingestion Pipeline** - the first stage of the Retrieval-Augmented Generation
(RAG) system. Without this, the assistant has no product knowledge at all.

```
Customer Query
     |
     v
[Document Ingestion Pipeline]  <-- YOU ARE HERE
     |
     v
[Vector Store + Retrieval]      (Topic 6)
     |
     v
[Chatbot with Context]          (Topics 6-7)
```

## ⚙️ Section 0: Environment Setup

I'll install the libraries we need and set up AWS credentials.
PyMuPDF is the fast PDF engine. BeautifulSoup handles HTML documents.
boto3 connects us to the S3 bucket where the Barclays documents live.

In [ ]:
# Install the document processing libraries we need for this topic.
# pymupdf: fast PDF text extraction (the underlying library is MuPDF, a C library)
# beautifulsoup4: HTML parsing and text extraction
# lxml: fast HTML/XML parser that BeautifulSoup uses as a backend
# boto3: AWS SDK for Python - lets us read from S3
# openai: used in Section 1 to show the naive raw-text approach failing on an LLM call
# sagemaker==2.232.1 is the last classic SDK release where get_execution_role lives at the top level
# numpy<2: pin to avoid breaking changes in numpy 2.x
!pip install -q "pymupdf==1.27.2.2" "beautifulsoup4==4.14.3" lxml boto3 "openai==2.32.0" "sagemaker==2.232.1" "numpy<2"

In [ ]:
import pymupdf          # PDF text extraction (replaces the older 'import fitz' pattern)
import boto3            # AWS SDK - for reading files from S3
import re               # Regular expressions for text cleaning
import textwrap         # Utility for wrapping text (used in display helpers)
import os
import getpass
from io import BytesIO  # For reading S3 binary response in memory (no disk write needed)
import sagemaker
from sagemaker import get_execution_role

# SageMaker session - gives us the IAM role that has S3 read access
sess = sagemaker.Session()
role = get_execution_role()

# AWS region where the S3 bucket lives
REGION = "us-east-2"

# For PRE-COURSE TESTING ONLY: change this to your personal test bucket.
# See setup/INSTRUCTOR_TEST_SETUP.ipynb for the test workflow.
# Restore "barclays-prompt-eng-data" before the course.
S3_BUCKET = "barclays-prompt-eng-data"

# The SageMaker execution role already has S3 read permissions for this bucket.
# We will be prompted for the OpenAI key in a later cell when the first API demo runs.
# The bulk of this topic is document processing with no LLM calls.
print(f"SageMaker role: ...{role[-30:]}")
print(f"S3 bucket: {S3_BUCKET}")
print(f"PyMuPDF version: {pymupdf.__version__}")

## 🏗️ What Are We Building Today?

The Barclays Product Knowledge Assistant needs to answer questions like:

- "What is the minimum repayment on a Barclays credit card?"
- "What fees apply to a personal loan early repayment?"
- "What are the terms for the Barclaycard Rewards card?"

To answer those accurately, the assistant needs to retrieve relevant text from real product
documents. Before any retrieval can happen, those documents need to be:

1. **Loaded** - pulled from S3 and read page by page
2. **Cleaned** - noise removed, whitespace normalized
3. **Chunked** - split into retrieval-sized fragments

Here is the difference between raw and preprocessed text fed to the same LLM:

**Before preprocessing (raw PDF dump):**
```
Barclaycard Rewards 2 INTEREST RATES AND CHARGES Purchase rate: 27.9% p.a.
variable Representative 27.9% APR variable Based on a credit limit of 1200
Your minimum payment See page 14 for full terms ...
```

**After preprocessing (clean chunk):**
```
The purchase rate on the Barclays Rewards credit card is 27.9% APR variable.
The minimum monthly repayment is whichever is greater: 1% of the outstanding
balance plus interest charges, or a fixed amount of GBP 25.
```

I am going to walk you through building the pipeline that produces the clean version.

## 📂 Section 1: Loading Documents from S3

### The problem: what happens if we skip proper document loading?

Let me show you what happens when I skip structured loading and just paste a snippet
of raw PDF dump directly into the LLM as context. This is what a lot of first-pass
integrations look like - a developer copies text out of a PDF viewer and drops it
straight into the prompt.

In [ ]:
# NAIVE APPROACH: raw PDF extraction pasted directly into the prompt.
# This simulates what many developers do when they first try to add documents to an LLM.
# Notice the noise: page numbers, repeated headers, broken sentences from line breaks.

from openai import OpenAI

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")
_client = OpenAI()

# This is what raw PyMuPDF output looks like BEFORE any cleaning or chunking.
# I copied this directly from a PDF extraction run - notice the artefacts.
raw_pdf_noise = """
Barclays Personal Loan   Important information   Page 1 of 12
PERSONAL LOAN AGREEMENT
Barclays Bank UK PLC
Early Repayment    If you repay  all or part of  your  loan early    we may charge  you
an  early  repayment  charge  (ERC)  of  up  to  58  days  interest.
Page 2  of  12   YOUR  LOAN  DETAILS
Amount borrowed:  see  schedule   APR  see  schedule   Duration  see  schedule
Page  2  of  12
"""

# Ask the LLM to answer from this raw context
naive_response = _client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
            "role": "system",
            "content": "You are a Barclays customer service assistant. Answer only from the provided context."
        },
        {
            "role": "user",
            "content": f"Context:\n{raw_pdf_noise}\n\nQuestion: What is the early repayment charge?"
        }
    ],
    max_tokens=150
)

print("LLM answer from raw noisy text:")
print(naive_response.choices[0].message.content)
print()
print("Notice how the answer is vague or hedged because the context is fragmented.")
print("The LLM is trying to make sense of broken sentences and page number artefacts.")

### How proper document loading fixes this

The fix is to load documents properly: pull the PDF binary from S3, open it with
PyMuPDF, iterate page by page, and join the pages into a single clean string.
That structured pass through the document preserves reading order and removes the
rendering artefacts we saw above.

```mermaid
graph TD
    A["S3 bucket\nbarclays-prompt-eng-data"] --> B["boto3 get_object\nreturns streaming body"]
    B --> C["BytesIO buffer\nin-memory bytes"]
    C --> D["pymupdf.open\nPyMuPDF opens PDF"]
    D --> E["Iterate pages\nfor page in doc"]
    E --> F["page.get_text\nraw text per page"]
    F --> G["Join pages\n'\\n'.join(pages)"]
    G --> H["Full document string\nready for cleaning"]
```

The diagram above shows the data flow: S3 bucket -> boto3 get_object -> BytesIO buffer
-> PyMuPDF open -> page iteration -> raw text per page -> joined document string.

In [ ]:
def load_pdf_from_s3(bucket: str, key: str, region: str = "us-east-2") -> str:
    """
    Load a PDF from S3 and return its full text as a single string.

    Args:
        bucket: S3 bucket name
        key: S3 object key (path inside the bucket)
        region: AWS region where the bucket lives

    Returns:
        Full text of the PDF, pages joined by double newline
    """
    # Create an S3 client. In SageMaker the execution role already has
    # the permissions we need - no credentials to pass manually.
    s3_client = boto3.client("s3", region_name=region)

    # get_object returns a dict. The file content is in the "Body" key
    # as a streaming object. We read it all into memory as bytes.
    response = s3_client.get_object(Bucket=bucket, Key=key)
    pdf_bytes = response["Body"].read()

    # BytesIO wraps the raw bytes so PyMuPDF can open them without writing
    # a temporary file to disk. This is cleaner in a notebook environment.
    pdf_buffer = BytesIO(pdf_bytes)

    # Open the PDF. pymupdf.open() accepts a stream argument for in-memory files.
    doc = pymupdf.open(stream=pdf_buffer, filetype="pdf")

    pages_text = []
    for page_num, page in enumerate(doc):
        # page.get_text() extracts the text layer of each page.
        # The default "text" mode preserves reading order with newlines.
        page_text = page.get_text("text")
        pages_text.append(page_text)

    # Join all pages. Double newline is a natural page boundary marker
    # that the cleaning step will use to detect paragraph breaks.
    full_text = "\n\n".join(pages_text)
    doc.close()
    return full_text


# Load the Barclays Personal Loan FAQ from the S3 bucket
print("Loading personal loan FAQ from S3...")
try:
    raw_text = load_pdf_from_s3(bucket=S3_BUCKET, key="barclays_personal_loan_faq.pdf")
    print(f"Loaded successfully. Character count: {len(raw_text):,}")
    print()
    print("First 400 characters of raw text:")
    print(raw_text[:400])
except Exception as e:
    # Fallback for when S3 is not available (e.g., running outside SageMaker).
    print(f"S3 load failed: {e}")
    print("Using fallback sample text for the demo.")
    raw_text = """Barclays Personal Loan FAQ\n\nPage 1\n\nQ: What is the early repayment charge?\n\nA: If you repay all or part of your loan early, we may charge you an early repayment charge (ERC) of up to 58 days interest on the amount you repay early.\n\nPage 2\n\nQ: How do I make a payment?\n\nA: You can make payments online via the Barclays app, by phone, or by direct debit set up at the time of the loan agreement.\n\nQ: What happens if I miss a payment?\n\nA: Missing a payment may affect your credit score and we may charge a late payment fee. Contact us immediately if you are having difficulty making payments.\n"""
    print(f"Fallback text loaded. Character count: {len(raw_text):,}")

### 🔬 Lab 1: Load a Second Document

I loaded the personal loan FAQ above. Now it is your turn to load the Barclaycard
Terms and Conditions document from the same bucket.

Your tasks:

1. Use `load_pdf_from_s3` to load `barclays_credit_card_tnc.pdf` from `S3_BUCKET`
2. Store the result in `credit_card_text`
3. Print the first 300 characters so you can see what the raw text looks like

If S3 is not available, the safety-net cell below will provide fallback text.

**Stretch**: Load a second document (`barclays_mortgage_summary.pdf`) and print the
character count. How does the length compare to the credit card document? What does
that tell you about how many chunks each will produce?

In [ ]:
# Lab 1 SOLUTION: Load the Barclaycard Terms and Conditions document from S3

# Step 1: call load_pdf_from_s3 with the credit card document key
# load_pdf_from_s3 handles the S3 connection, BytesIO buffer, and PyMuPDF iteration
# for us - we just need to pass the bucket and key.
try:
    credit_card_text = load_pdf_from_s3(bucket=S3_BUCKET, key="barclays_credit_card_tnc.pdf")
except Exception as e:
    # When running outside SageMaker (no S3), fall back to the sample text.
    # In class this branch should not fire - S3 is pre-loaded by the instructor.
    print(f"S3 load failed: {e} - using fallback text")
    credit_card_text = """Barclaycard Terms and Conditions\n\nSection 1: Your Agreement\n\nThis agreement is between you and Barclays Bank UK PLC.\n\nSection 2: Interest Rates\n\nThe purchase rate is 27.9% APR variable. Cash advance rate is 29.9% APR variable.\n\nSection 3: Minimum Repayment\n\nYour minimum monthly repayment is the greater of: 1% of your outstanding balance plus interest charges for the month, or GBP 25, or your full balance if it is less than GBP 25.\n\nSection 4: Late Payment\n\nIf we do not receive at least your minimum payment by the due date we will charge a late payment fee of GBP 12.\n"""

# Step 2: print the first 300 characters so we can inspect the raw PDF output
print("First 300 characters of raw credit card text:")
print(credit_card_text[:300])
print()

# Verification
if credit_card_text is not None and len(credit_card_text) > 100:
    print("[OK] Document loaded successfully.")
    print(f"[OK] Character count: {len(credit_card_text):,}")
else:
    print("[ ] credit_card_text is not set or too short. Check the function call above.")

In [ ]:
# Safety-net: if you did not finish Lab 1 above, run this cell so the rest of the
# notebook still works. If you completed Lab 1 successfully, SKIP this cell.
if credit_card_text is None:
    print("Using safety-net so the notebook can continue.")
    credit_card_text = """Barclaycard Terms and Conditions\n\nSection 1: Your Agreement\n\nThis agreement is between you and Barclays Bank UK PLC.\n\nSection 2: Interest Rates\n\nThe purchase rate is 27.9% APR variable. Cash advance rate is 29.9% APR variable.\n\nSection 3: Minimum Repayment\n\nYour minimum monthly repayment is the greater of: 1% of your outstanding balance plus interest charges for the month, or GBP 25, or your full balance if it is less than GBP 25.\n\nSection 4: Late Payment\n\nIf we do not receive at least your minimum payment by the due date we will charge a late payment fee of GBP 12.\n"""
    print(f"Safety-net credit_card_text set. Character count: {len(credit_card_text):,}")

## 🧹 Section 2: Text Cleaning

### The problem: PDF extraction is noisy even after loading

Loading the PDF properly gives us text, but PDF extraction has well-known artefacts.
Here is what the raw text typically contains that we need to remove:

- **Repeated headers and footers** ("Barclays Bank UK PLC  Page 3 of 12") on every page
- **Excessive whitespace** from PDF column alignment leaving multiple spaces between words
- **Hyphenation artefacts** - words split across lines like "repay-\nment"
- **Boilerplate separators** - lines that are just dashes or dots used as visual dividers
- **Irregular newlines** - single newlines mid-sentence from PDF line breaks

Let me show you what the retrieval pipeline sees without cleaning.

In [ ]:
# NAIVE APPROACH: pass raw extracted text directly to the LLM as a context chunk.
# Watch what happens when the chunk contains page headers, inconsistent whitespace,
# and hyphenated line breaks.

# Simulate what a chunk from raw (uncleaned) PDF extraction looks like.
noisy_chunk = """Barclays Bank UK PLC   Page 4 of 12   Personal Loan Agreement
Early Re-
payment Charge (ERC)

If  you  repay  all  or  part   of  your   loan  early   we  may  charge  you  an
early  repayment  charge  (ERC).   The  ERC  is  up  to  58   days  interest   on
the  amount  repaid  early.

Barclays Bank UK PLC   Page 4 of 12   Personal Loan Agreement"""

noisy_response = _client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
            "role": "system",
            "content": "You are a Barclays customer service assistant. Answer only from the context."
        },
        {
            "role": "user",
            "content": f"Context:\n{noisy_chunk}\n\nQuestion: How is the early repayment charge calculated?"
        }
    ],
    max_tokens=120
)

print("Answer from noisy chunk:")
print(noisy_response.choices[0].message.content)
print()
print("The LLM had to parse around the page header, the hyphenated 'Re-payment',")
print("and the doubled whitespace. Cleaning eliminates all of this before retrieval.")

### How the cleaning pipeline fixes this

A cleaning pipeline is a series of deterministic text transformations applied in order.
Each transformation targets one category of noise. The order matters: I fix hyphenated
line breaks before normalizing whitespace, because the fix joins two halves of a word
and the whitespace normalizer then cleans up any leftover spaces.

```mermaid
graph TD
    A["Raw PDF text\nhyphenation, headers, noise"] --> B["Dehyphenate\nrejoin split words"]
    B --> C["Remove headers/footers\nstrip page numbers, titles"]
    C --> D["Collapse whitespace\nnormalize spaces and newlines"]
    D --> E["Strip boilerplate\nremove repeated legal lines"]
    E --> F["Normalize sentence breaks\nensure single space after period"]
    F --> G["Clean text\nready for chunking"]
```

The diagram above shows the ordered sequence: dehyphenate -> remove headers/footers
-> collapse whitespace -> strip boilerplate lines -> normalize sentence breaks.

In [ ]:
def clean_pdf_text(text: str) -> str:
    """
    Clean raw PDF-extracted text for use in a RAG pipeline.

    Steps applied in order:
    1. Fix hyphenated line breaks (re-\npayment -> repayment)
    2. Remove repeated page header/footer patterns
    3. Collapse multiple spaces and tabs to a single space
    4. Normalize line endings (CRLF and multiple newlines to double newline)
    5. Strip lines that are purely decorative (all dashes, dots, or digits)
    6. Strip leading/trailing whitespace from the whole document
    """
    # Step 1: Fix hyphenated line breaks.
    # PDF layout engines break long words at line ends with a hyphen.
    # "re-\npayment" -> "repayment"
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # Step 2: Remove page header and footer boilerplate.
    # Barclays PDFs have patterns like "Barclays Bank UK PLC   Page N of N"
    # on every page. These carry no content - remove them.
    text = re.sub(r"Barclays Bank UK PLC\s+Page\s+\d+\s+of\s+\d+[^\n]*\n", "", text)

    # Step 3: Collapse multiple spaces and tabs to a single space.
    # PDF column alignment leaves multiple spaces between words.
    text = re.sub(r"[ \t]+", " ", text)

    # Step 4: Normalize line endings.
    # Convert Windows CRLF to Unix LF, then collapse 3+ newlines to 2.
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Step 5: Remove boilerplate-only lines.
    # Lines that are just dashes, dots, or digits are visual separators with no content.
    text = re.sub(r"^\s*[-_.=*]{3,}\s*$", "", text, flags=re.MULTILINE)
    text = re.sub(r"^\s*\d+\s*$", "", text, flags=re.MULTILINE)

    # Step 6: Final strip
    return text.strip()


# Apply the cleaning function to the personal loan text we loaded
cleaned_text = clean_pdf_text(raw_text)

print("Cleaning complete.")
print(f"Raw length:     {len(raw_text):,} characters")
print(f"Cleaned length: {len(cleaned_text):,} characters")
print(f"Characters removed: {len(raw_text) - len(cleaned_text):,}")
print()
print("First 400 characters of cleaned text:")
print(cleaned_text[:400])

### 🔬 Lab 2: Clean the Credit Card Document

You loaded `credit_card_text` in Lab 1. Now clean it using `clean_pdf_text`.

Your tasks:

1. Apply `clean_pdf_text` to `credit_card_text` and store the result in `cleaned_credit_card_text`
2. Print a comparison: raw character count vs cleaned character count
3. Print the first 300 characters of the cleaned version

**Stretch**: Add a custom cleaning step to `clean_pdf_text` that removes lines shorter
than 10 characters (they are often noise - stray page numbers or single words from
headers). How does this change the character count?

In [ ]:
# Lab 2 SOLUTION: Clean the credit card Terms and Conditions text

# Step 1: apply clean_pdf_text to credit_card_text
# clean_pdf_text runs the 6-step regex pipeline: dehyphenate, remove headers,
# collapse whitespace, normalize newlines, strip boilerplate lines, final strip.
cleaned_credit_card_text = clean_pdf_text(credit_card_text)

# Step 2: print a comparison so we can see how much noise was removed
print("Character count comparison:")
print(f"  Raw:     {len(credit_card_text):,} characters")
print(f"  Cleaned: {len(cleaned_credit_card_text):,} characters")
print(f"  Removed: {len(credit_card_text) - len(cleaned_credit_card_text):,} characters ({(len(credit_card_text) - len(cleaned_credit_card_text)) / max(len(credit_card_text), 1) * 100:.1f}%)")
print()

# Step 3: print the first 300 characters of the cleaned version
# This should look noticeably cleaner - no doubled spaces, no page headers
print("First 300 characters of cleaned credit card text:")
print(cleaned_credit_card_text[:300])
print()

# Common mistake: calling clean_pdf_text on already-cleaned text adds no harm but is
# wasteful. Always clean once on the raw extracted text before chunking.

# Verification
if cleaned_credit_card_text is not None and len(cleaned_credit_card_text) > 50:
    print("[OK] Cleaned text produced successfully.")
    print(f"[OK] Cleaned length: {len(cleaned_credit_card_text):,}")
else:
    print("[ ] cleaned_credit_card_text is not set or empty. Check Step 1 above.")

## ✂️ Section 3: Chunking Strategies

### The problem: why the full document in one prompt does not scale

I have clean text now. Why not just dump the whole document into the LLM prompt?
Three reasons:

1. **Token limits**: GPT-4o has a 128K context window. A full set of Barclays product
   PDFs can easily exceed that. Even if it fits today, cost compounds across thousands
   of customer queries per day.

2. **Retrieval precision**: In Topics 6-7 we embed each chunk and search for the most
   relevant ones. If each "chunk" is the whole 40-page document, every query returns
   the same thing. Small, precise chunks let retrieval surface the specific paragraph
   that answers the question.

3. **Hallucination risk**: The more irrelevant text the LLM sees around the answer, the
   more likely it is to mix in details from the wrong section. Focused chunks reduce this.

The question is: how big should each chunk be, and should they overlap? Let me show
you what happens when chunk size is badly calibrated.

In [ ]:
# NAIVE APPROACH: chunk size too small (50 chars) - chunks lose all context.
# This simulates what happens when a developer picks an arbitrary tiny chunk size.

sample_text = """The Barclays Rewards credit card has a purchase rate of 27.9% APR variable.
The minimum monthly repayment is the greater of 1% of your outstanding balance plus
interest charges for the month, or GBP 25, whichever is higher. If your balance is
less than GBP 25 you must repay the full balance. Cash advances are charged at a
higher rate of 29.9% APR variable and interest accrues from the date of the transaction."""

# Split into tiny 50-character chunks - too small to carry context
tiny_chunks = [sample_text[i:i+50] for i in range(0, len(sample_text), 50)]

print(f"Number of chunks: {len(tiny_chunks)}")
print()
print("Chunks preview (notice how most are meaningless fragments):")
for idx, chunk in enumerate(tiny_chunks[:5]):
    print(f"  Chunk {idx}: '{chunk}'")

print()
print("These fragments are useless for retrieval. 'APR variable.' on its own")
print("tells the LLM nothing about which product, which rate, or what it applies to.")

### Fixed-size chunking with overlap - the practical default

The best-validated default for most RAG pipelines in 2026 is recursive character
splitting at 400-512 tokens (roughly 1600-2048 characters) with 10-20% overlap.

Overlap matters because a key sentence might sit right at a chunk boundary. With no
overlap, that sentence is split and neither chunk has the full meaning. With overlap,
each chunk contains some of its neighbour's content, so the boundary sentence appears
intact in at least one chunk.

```mermaid
graph TD
    A["Clean document text\ne.g. 10,000 characters"] --> B["Splitter\nchunk_size=1800\noverlap=200"]
    B --> C["Chunk 1\nchars 0 to 1800"]
    B --> D["Chunk 2\nchars 1600 to 3400\noverlap with Chunk 1"]
    B --> E["Chunk 3\nchars 3200 to 5000\noverlap with Chunk 2"]
    B --> F["... more chunks"]
    C --> G["Chunk list\nready for embedding"]
    D --> G
    E --> G
    F --> G
```

The diagram above shows fixed-size chunking with an overlap window: chunk N ends at
character position X, chunk N+1 starts at X minus the overlap size, so both chunks
share a region of text around the boundary.

In [ ]:
def chunk_text(text: str, chunk_size: int = 1500, overlap: int = 200) -> list:
    """
    Split text into overlapping fixed-size chunks.

    chunk_size of 1500 characters is approximately 375-400 tokens (GPT-4o tokenizer
    averages about 4 characters per token for English text).
    overlap of 200 characters (~50 tokens) is about 13% - within the recommended
    10-20% range validated in 2026 RAG benchmarks.

    Args:
        text: cleaned text string to split
        chunk_size: maximum characters per chunk
        overlap: characters of overlap between adjacent chunks

    Returns:
        List of chunk strings, each at most chunk_size characters
    """
    if not text:
        return []

    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size

        # If we are not at the very end, try to break at a sentence boundary.
        # A hard cut mid-sentence is worse than a cut at a period or newline.
        if end < len(text):
            boundary = max(
                text.rfind(". ", start, end),
                text.rfind(".\n", start, end),
                text.rfind("\n\n", start, end)
            )
            # Only use the boundary if it is at least 50% into the chunk window
            if boundary > start + chunk_size // 2:
                end = boundary + 1

        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)

        # Step forward by (chunk_size - overlap) to create the overlap window.
        start += chunk_size - overlap

    return chunks


# Apply chunking to the cleaned personal loan text
chunks = chunk_text(cleaned_text, chunk_size=1500, overlap=200)

print(f"Chunking complete.")
print(f"Total chunks: {len(chunks)}")
print(f"Average chunk length: {sum(len(c) for c in chunks) / max(len(chunks), 1):.0f} characters")
print()
print("Chunk 0 preview (first 300 chars):")
print(chunks[0][:300] if chunks else "(no chunks)")
print()
print("Chunk 1 preview - notice the overlap with Chunk 0:")
print(chunks[1][:300] if len(chunks) > 1 else "(only one chunk)")

### 🔬 Lab 3 (Hard): Build a Full Preprocessing Pipeline

This is the hard lab for Day 1. Wire together everything we built into a single reusable
function, then run a controlled experiment to see how chunk size affects the output.

**Part A: Build `preprocess_document()`**

Write a function `preprocess_document(bucket, key, chunk_size, overlap)` that:
1. Loads the PDF from S3 using `load_pdf_from_s3()`
2. Cleans the text using `clean_pdf_text()`
3. Chunks the cleaned text using `chunk_text()` with the given parameters
4. Returns the list of chunks

**Part B: Chunk Size Experiment**

Call `preprocess_document()` three times on the personal loan PDF (`barclays_personal_loan_faq.pdf`),
using chunk sizes 512, 1024, and 1500 (all with overlap=200). Store results in:
- `chunks_512`
- `chunks_1024`
- `chunks_1500`

Then answer in a print statement: which chunk size gives the most chunks? Which gives
the longest average chunk? Which would you pick for production and why?

**Stretch challenge**: Add a `min_chunk_length` parameter (default 100) that filters out
chunks shorter than that threshold. This prevents near-empty chunks from polluting the
vector store when a page is mostly a header.

**Homework Extension**: Real pipelines handle multiple formats. Design a
`load_document(source, format)` dispatcher that routes to `load_pdf_from_s3()`,
`load_html()`, or raises `ValueError` for unsupported formats. Consider: what metadata
(source URL, page number, document title) would you attach to each chunk?

In [ ]:
# Lab 3 SOLUTION: Full preprocessing pipeline and chunk size experiment

def preprocess_document(bucket: str, key: str, chunk_size: int = 1500, overlap: int = 200) -> list:
    """
    Full preprocessing pipeline: load -> clean -> chunk.

    This is the function students in Topic 6 will call to populate ChromaDB.
    Keeping load/clean/chunk wired together here means the caller only needs
    to know the S3 path and the desired chunk geometry.
    """
    # Step 1: load the raw PDF text from S3
    raw = load_pdf_from_s3(bucket=bucket, key=key)

    # Step 2: remove PDF noise before chunking
    # Cleaning before chunking is important: if we chunk first, noise like
    # "Page N of N" can appear at the start of a chunk, wasting token budget
    # and confusing retrieval embeddings.
    cleaned = clean_pdf_text(raw)

    # Step 3: split into overlapping chunks and return
    return chunk_text(cleaned, chunk_size=chunk_size, overlap=overlap)


# Part B: Chunk size experiment
# We run the same document through three different chunk geometries and compare.
# S3 fallback: if the bucket is not available, use cleaned_text from earlier.
_loan_key = "barclays_personal_loan_faq.pdf"
try:
    chunks_512  = preprocess_document(S3_BUCKET, _loan_key, chunk_size=512,  overlap=200)
    chunks_1024 = preprocess_document(S3_BUCKET, _loan_key, chunk_size=1024, overlap=200)
    chunks_1500 = preprocess_document(S3_BUCKET, _loan_key, chunk_size=1500, overlap=200)
except Exception:
    chunks_512  = chunk_text(cleaned_text, chunk_size=512,  overlap=200)
    chunks_1024 = chunk_text(cleaned_text, chunk_size=1024, overlap=200)
    chunks_1500 = chunk_text(cleaned_text, chunk_size=1500, overlap=200)

# Print comparison table
print("Chunk size experiment results:")
print(f"{'Size':>6}  {'Count':>6}  {'Avg len':>8}  {'Max len':>8}")
for label, clist in [("512", chunks_512), ("1024", chunks_1024), ("1500", chunks_1500)]:
    avg = sum(len(c) for c in clist) / max(len(clist), 1)
    mx  = max(len(c) for c in clist) if clist else 0
    print(f"{label:>6}  {len(clist):>6}  {avg:>8.0f}  {mx:>8}")

print()
print("Recommendation: 1500 chars with 200 overlap is the best default for Barclays product FAQs.")
print("Reasoning: FAQ answers are typically 2-4 sentences (300-600 chars). A 1500-char chunk")
print("captures the full question-answer pair plus some surrounding context without wasting")
print("embedding capacity on multiple unrelated topics per chunk.")
print()
# Common mistake: overlap larger than chunk_size causes infinite loops.
# chunk_text guards against this with the start += chunk_size - overlap step.
# If chunk_size=200 and overlap=250, every iteration step would be negative.
# Always keep overlap < chunk_size.

# Verification
if chunks_512 is not None and chunks_1024 is not None and chunks_1500 is not None:
    print("[OK] All three chunk sets produced.")
    print(f"  chunks_512:  {len(chunks_512)} chunks")
    print(f"  chunks_1024: {len(chunks_1024)} chunks")
    print(f"  chunks_1500: {len(chunks_1500)} chunks")
else:
    print("[ ] One or more chunk sets are None. Check the function and the three calls above.")

In [ ]:
# Safety-net: if you did not finish Lab 3 above, run this cell so the wrap-up
# still works. If you completed Lab 3 successfully, SKIP this cell.
if chunks_512 is None or chunks_1024 is None or chunks_1500 is None:
    print("Using safety-net implementation so the notebook can continue.")

    def preprocess_document(bucket: str, key: str, chunk_size: int = 1500, overlap: int = 200) -> list:
        raw = load_pdf_from_s3(bucket=bucket, key=key)
        cleaned = clean_pdf_text(raw)
        return chunk_text(cleaned, chunk_size=chunk_size, overlap=overlap)

    try:
        chunks_512  = preprocess_document(S3_BUCKET, "barclays_personal_loan_faq.pdf", chunk_size=512,  overlap=200)
        chunks_1024 = preprocess_document(S3_BUCKET, "barclays_personal_loan_faq.pdf", chunk_size=1024, overlap=200)
        chunks_1500 = preprocess_document(S3_BUCKET, "barclays_personal_loan_faq.pdf", chunk_size=1500, overlap=200)
    except Exception:
        # Fallback when S3 is not available
        chunks_512  = chunk_text(cleaned_text, chunk_size=512,  overlap=200)
        chunks_1024 = chunk_text(cleaned_text, chunk_size=1024, overlap=200)
        chunks_1500 = chunk_text(cleaned_text, chunk_size=1500, overlap=200)

    print(f"Safety-net chunks_512:  {len(chunks_512)} chunks")
    print(f"Safety-net chunks_1024: {len(chunks_1024)} chunks")
    print(f"Safety-net chunks_1500: {len(chunks_1500)} chunks")

## ✅ Wrap-Up: What We Built Today

In this topic I built the **Document Ingestion Pipeline** - the first stage of the Barclays
Product Knowledge Assistant.

**What I built:**

| Function | What it does |
|---|---|
| `load_pdf_from_s3(bucket, key)` | Pulls a PDF from S3 and returns full text |
| `clean_pdf_text(text)` | Removes headers, normalizes whitespace, fixes hyphenation |
| `chunk_text(text, chunk_size, overlap)` | Splits into overlapping fixed-size chunks |
| `preprocess_document(bucket, key, chunk_size, overlap)` | Full pipeline in one call |

**Chunk size rules of thumb:**

- 512 chars (~128 tokens): high precision, low context. Good when queries are very specific.
- 1024 chars (~256 tokens): balanced default for most RAG tasks.
- 1500 chars (~375 tokens): more context per chunk, better for multi-sentence answers.
- Overlap at 10-20% of chunk size reduces boundary-cut information loss.

**What comes next:**

- **Topic 3**: We use the `chunks` variable from this notebook to build a chatbot that
  loads document context into the system prompt. You will see the first version of the
  Barclays Product Knowledge Assistant respond to customer queries.
- **Topics 6-7**: The `preprocess_document()` pipeline feeds directly into a ChromaDB vector
  store where each chunk becomes an embedding. That is when retrieval becomes semantic rather
  than keyword-based.

**Homework Extensions:**

1. Add HTML document support: extract clean text from a Barclays web page using
   BeautifulSoup and `soup.get_text(separator=" ", strip=True)`. Wire it into a
   `load_html(url)` function and integrate with `preprocess_document()`.
2. Add chunk metadata: return a list of dicts `{"text": chunk, "source": key, "chunk_id": i}`
   instead of plain strings. In Topics 6-7 this metadata lets you cite sources in answers.
3. Benchmark your cleaning: compute the "noise ratio" (characters removed / original length)
   for 3 different Barclays PDFs. Are some document types noisier than others?